In [0]:
# Loading data from Ingested files to the Bronze schema in nycyellow catalog

from pyspark.sql.functions import col, current_timestamp

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "/Volumes/nycyellow/landing/raw/schema_location/")
    .load("/Volumes/nycyellow/landing/raw/*.parquet")
    .withColumn("source_file_name", col("_metadata.file_name"))
    .withColumn("file_load_time", col("_metadata.file_modification_time"))
    .withColumn("bronze_ingestion_time", current_timestamp())
    .writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/nycyellow/landing/raw/checkpoint/_checkpoint_batch")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("nycyellow.bronze.nyctripdata_raw")
)
        

In [0]:
# Reading the static zone lookup table as well into the bronze layer 
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/nycyellow/landing/raw/zone_lookup/")
display(df)

df.writeTo("nycyellow.bronze.nyctripdata_zone_lookup").createOrReplace()

In [0]:
# Checking to see if the data has loaded into the table or not
df= spark.read.table("nycyellow.bronze.nyctripdata_raw")
display(df)

In [0]:
# Checking to see if the data has read into the lookup table 
df = spark.read.table("nycyellow.bronze.nyctripdata_zone_lookup")

display(df)